## In Class 04/28


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score

In [8]:
irrigation_data=pd.read_csv("../KagglePlaygroundS6E4/playground-series-s6e4/train.csv")

irrigation_data.head()

irrigation_X = irrigation_data.drop(columns=["Irrigation_Need", "id"])
irrigation_y = irrigation_data["Irrigation_Need"]

irrigation_X_train, irrigation_X_test, irrigation_y_train, irrigation_y_test = train_test_split(
    irrigation_X, irrigation_y, test_size=0.2, random_state=42, stratify=irrigation_y
)

irrigation_X_train.head()



,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
344420,Silt,5.66,51.38,0.46,1.46,33.21,66.34,775.35,6.23,17.09,Maize,Flowering,Zaid,Canal,Reservoir,13.63,Yes,66.62,Central
29017,Loamy,6.14,8.03,1.14,3.34,20.85,74.29,2374.11,7.52,1.94,Sugarcane,Vegetative,Rabi,Rainfed,Groundwater,5.56,Yes,92.24,West
570499,Sandy,5.77,59.04,1.08,0.55,14.44,37.99,2137.79,5.05,10.46,Rice,Sowing,Rabi,Drip,Reservoir,0.78,Yes,92.31,South
620696,Sandy,5.06,63.19,1.59,1.10,23.58,58.25,1226.78,4.54,1.15,Wheat,Harvest,Kharif,Canal,Reservoir,2.67,No,113.36,West
7445,Clay,6.28,35.94,1.11,1.57,34.30,71.04,2266.39,8.47,16.81,Maize,Vegetative,Kharif,Drip,Groundwater,6.74,Yes,96.90,North


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report

# Identify column types
num_cols = irrigation_X_train.select_dtypes(include=["number"]).columns
cat_cols = irrigation_X_train.select_dtypes(exclude=["number"]).columns

# Preprocess + GaussianNB pipeline
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

gnb_pipe = Pipeline(
    steps=[
        ("prep", preprocess),
        ("model", GaussianNB()),
    ]
)

gnb_pipe.fit(irrigation_X_train, irrigation_y_train)

# Predicted probabilities + baseline default prediction (argmax)
y_prob = gnb_pipe.predict_proba(irrigation_X_test)
classes = gnb_pipe.named_steps["model"].classes_
y_pred_baseline = classes[np.argmax(y_prob, axis=1)]

print("Baseline classification report (default max-prob rule):")
print(classification_report(irrigation_y_test, y_pred_baseline, digits=4))
print("Baseline balanced accuracy:", round(balanced_accuracy_score(irrigation_y_test, y_pred_baseline), 6))

Baseline classification report (default max-prob rule):
              precision    recall  f1-score   support

        High     0.5239    0.7761    0.6255      4202
         Low     0.8750    0.7970    0.8342     73983
      Medium     0.6977    0.7644    0.7295     47815

    accuracy                         0.7840    126000
   macro avg     0.6989    0.7792    0.7297    126000
weighted avg     0.7960    0.7840    0.7875    126000

Baseline balanced accuracy: 0.779168


In [11]:
# Threshold tuning for rare/important class: High
focus_class = "High"
focus_idx = np.where(classes == focus_class)[0][0]

threshold_grid = np.arange(0.10, 0.91, 0.05)
rows = []

for thr in threshold_grid:
    # Start from non-High best class, then force High when prob is above threshold
    non_focus_prob = y_prob.copy()
    non_focus_prob[:, focus_idx] = -1.0
    y_pred_thr = classes[np.argmax(non_focus_prob, axis=1)]
    y_pred_thr[y_prob[:, focus_idx] >= thr] = focus_class

    rows.append(
        {
            "threshold": round(float(thr), 2),
            "balanced_accuracy": balanced_accuracy_score(irrigation_y_test, y_pred_thr),
            "precision_high": precision_score(irrigation_y_test, y_pred_thr, labels=[focus_class], average="macro", zero_division=0),
            "recall_high": recall_score(irrigation_y_test, y_pred_thr, labels=[focus_class], average="macro", zero_division=0),
            "f1_high": f1_score(irrigation_y_test, y_pred_thr, labels=[focus_class], average="macro", zero_division=0),
        }
    )

threshold_results = (
    pd.DataFrame(rows)
    .sort_values(["recall_high", "precision_high"], ascending=[False, False])
    .reset_index(drop=True)
)
best_threshold = float(threshold_results.loc[0, "threshold"])

# Build predictions at best threshold for full report
non_focus_prob = y_prob.copy()
non_focus_prob[:, focus_idx] = -1.0
y_pred_best_thr = classes[np.argmax(non_focus_prob, axis=1)]
y_pred_best_thr[y_prob[:, focus_idx] >= best_threshold] = focus_class

print(f"Best threshold by recall for class '{focus_class}': {best_threshold}")
print("\nTop threshold settings:")
display(threshold_results.head(10))

print("\nClassification report at best threshold:")
print(classification_report(irrigation_y_test, y_pred_best_thr, digits=4))

baseline_metrics = {
    "balanced_accuracy": balanced_accuracy_score(irrigation_y_test, y_pred_baseline),
    "precision_high": precision_score(irrigation_y_test, y_pred_baseline, labels=[focus_class], average="macro", zero_division=0),
    "recall_high": recall_score(irrigation_y_test, y_pred_baseline, labels=[focus_class], average="macro", zero_division=0),
    "f1_high": f1_score(irrigation_y_test, y_pred_baseline, labels=[focus_class], average="macro", zero_division=0),
}

best_row = threshold_results.iloc[0]
comparison = pd.DataFrame(
    [
        {
            "setting": "baseline_default",
            **baseline_metrics,
        },
        {
            "setting": f"threshold_{best_threshold}",
            "balanced_accuracy": best_row["balanced_accuracy"],
            "precision_high": best_row["precision_high"],
            "recall_high": best_row["recall_high"],
            "f1_high": best_row["f1_high"],
        },
    ]
)

print("\nBaseline vs thresholded-best comparison:")
display(comparison)

Best threshold by recall for class 'High': 0.1

Top threshold settings:


,threshold,balanced_accuracy,precision_high,recall_high,f1_high
0,0.10,0.769799,0.301615,0.857687,0.446288
1,0.15,0.772589,0.331614,0.841504,0.475748
2,0.20,0.774900,0.359522,0.829843,0.501691
3,0.25,0.776246,0.384444,0.819848,0.523437
4,0.30,0.778030,0.411453,0.812232,0.546211
5,0.35,0.778398,0.435957,0.802713,0.565039
6,0.40,0.778370,0.461741,0.792718,0.583567
7,0.45,0.779246,0.491952,0.785578,0.605022
8,0.50,0.779128,0.525137,0.775583,0.626249
9,0.55,0.778546,0.556747,0.765826,0.644761



Classification report at best threshold:
              precision    recall  f1-score   support

        High     0.3016    0.8577    0.4463      4202
         Low     0.8750    0.7970    0.8342     73983
      Medium     0.6709    0.6547    0.6627     47815

    accuracy                         0.7450    126000
   macro avg     0.6158    0.7698    0.6477    126000
weighted avg     0.7784    0.7450    0.7562    126000


Baseline vs thresholded-best comparison:


,setting,balanced_accuracy,precision_high,recall_high,f1_high
0,baseline_default,0.779168,0.523855,0.776059,0.625492
1,threshold_0.1,0.769799,0.301615,0.857687,0.446288


### GaussianNB Thresholding Discussion

I focused on the **`High` irrigation need** class because it is the rare/important class and missing it is bad for farming.

I looked at a bunch of different thresholds for high, and the lowest were obviously the best performing in terms of recall

The "best" threshold specifically for recall is 0.10. However, if you want more balance and not too many watered grounds that you dont actually need to water, you probably should consider using a larger threshold like 0.25

So there are obvious tradeoffs:

- Lower thresholds increase **recall for `High`** (fewer missed High cases) but reduce precision.
- Higher thresholds usually improve **precision for `High`** but reduce recall.

Also another thing to notice is the balanced accuracy didn't really change at all with the different thresholds.


Compared with stronger models from the Kaggle workflow (like boosted trees / stacked ensemble), GaussianNB is way simpler and faster but less accurate overall. Using this threshold tuning can make GaussianNB more practical when there is a need for certain class to be prioritized over others.